In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers
!pip install faiss-gpu-cu12

In [ ]:
import torch
import pandas as pd
import numpy as np
import faiss
import json
import gc
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm

def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    top_k_preds = set(retrieved_ids[:k])
    correct_preds = top_k_preds.intersection(ground_truth_ids)
    hit_val = 1 if len(correct_preds) > 0 else 0
    if len(ground_truth_ids) > 0:
        recall_val = len(correct_preds) / len(ground_truth_ids)
    else:
        recall_val = 0.0
    return hit_val, recall_val

def evaluate_retrieval_rag_final_optimized(
    test_data_path, 
    corpus_data_path, 
    faiss_index_path, 
    cross_encoder_name='BAAI/bge-reranker-v2-m3',
    k_values=[1, 5, 10],
    top_k_rerank=10,  
    batch_process_size=100 
    ):
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Load Models
    print("\n--- Loading Models ---")
    bi_encoder = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)
    if device == 'cuda': bi_encoder.half() 
    
    print(f"Loading Cross-Encoder: {cross_encoder_name}")
    cross_encoder = CrossEncoder(cross_encoder_name, max_length=512, device=device)
    if device == 'cuda':
        cross_encoder.model.half()

    # Load Corpus Data & Map ID
    print(f"\n--- Loading Corpus Data ---")
    df_chunks = pd.read_csv(corpus_data_path, encoding='utf-8')
    
    # Map index -> ID & Text
    all_chunk_ids = df_chunks['id'].astype(str).values
    
    print("Pre-processing corpus text...")
    if 'title' in df_chunks.columns and 'text' in df_chunks.columns:
        titles = df_chunks['title'].astype(str).values
        texts = df_chunks['text'].astype(str).values
        all_chunk_texts = ["Title: " + t + ". Content: " + txt for t, txt in zip(titles, texts)]
    else:
        all_chunk_texts = df_chunks['text'].astype(str).tolist()

    print(f"Loading FAISS index...")
    index = faiss.read_index(faiss_index_path)

    # Load Test Data 
    print(f"\n--- Loading Test Data ---")
    queries = []
    ground_truths = []
    
    with open(test_data_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)
            q = item.get("question", "").strip()
            rels = item.get("relevant_articles", [])
            
            if q:
                queries.append(q)
                gt_ids = set()
                for article in rels:
                    if "ans_id" in article:
                        for a_id in article["ans_id"]:
                            gt_ids.add(str(a_id))
                ground_truths.append(gt_ids)
        
    total_queries = len(queries)
    print(f"Total Queries: {total_queries}")

    print(f"\n--- Stage 1: Retrieval (Bi-Encoder) ---")
    query_embeddings = bi_encoder.encode(queries, convert_to_numpy=True, batch_size=512, show_progress_bar=True)
    query_embeddings = query_embeddings.astype('float32')
    faiss.normalize_L2(query_embeddings)
    
    print(f"Searching Top {top_k_rerank} candidates...")
    D, I = index.search(query_embeddings, top_k_rerank)

    print(f"\n--- Stage 2: Reranking & Eval (Processing {batch_process_size} queries at a time) ---")
    
    metrics_retriever = {'hit': {k: 0 for k in k_values}, 'recall': {k: 0.0 for k in k_values}}
    metrics_reranker = {'hit': {k: 0 for k in k_values}, 'recall': {k: 0.0 for k in k_values}}
    
    # Vòng lặp xử lý từng lô câu hỏi
    for i in tqdm(range(0, total_queries, batch_process_size), desc="Batch Processing"):
        # Lấy slice cho batch hiện tại
        batch_end = min(i + batch_process_size, total_queries)
        batch_indices = range(i, batch_end)
        
        batch_pairs = []
        batch_meta = [] 
        
        for q_idx in batch_indices:
            q_text = queries[q_idx]
            gt_ids = ground_truths[q_idx]
            
            retrieved_indices = I[q_idx]
            
            current_doc_ids = []
            current_doc_texts = []
            
            for doc_idx in retrieved_indices:
                if doc_idx != -1 and doc_idx < len(all_chunk_ids):
                    current_doc_ids.append(all_chunk_ids[doc_idx])
                    current_doc_texts.append(all_chunk_texts[doc_idx])
            
            # 1. Đánh giá ngay Bi-Encoder (Retriever) cho query này
            if gt_ids:
                for k in k_values:
                    h, r = calculate_metrics(current_doc_ids, gt_ids, k)
                    metrics_retriever['hit'][k] += h
                    metrics_retriever['recall'][k] += r
            
            # 2. Gom cặp để Rerank
            if current_doc_texts:
                for doc_text, doc_id in zip(current_doc_texts, current_doc_ids):
                    batch_pairs.append([q_text, doc_text])
                
                # Lưu thông tin để tái cấu trúc sau khi predict
                # q_idx: chỉ số câu hỏi
                # count: số lượng doc của câu hỏi này trong batch_pairs
                # doc_ids: danh sách id gốc
                batch_meta.append({
                    'q_idx': q_idx,
                    'count': len(current_doc_texts),
                    'doc_ids': current_doc_ids
                })

        if batch_pairs:
            scores = cross_encoder.predict(batch_pairs, batch_size=128, show_progress_bar=False)
            
            # Map điểm số ngược lại từng câu hỏi và đánh giá Reranker
            cursor = 0
            for meta in batch_meta:
                q_idx = meta['q_idx']
                count = meta['count']
                doc_ids = meta['doc_ids']
                gt_ids = ground_truths[q_idx]
                
                # Lấy điểm tương ứng
                q_scores = scores[cursor : cursor + count]
                cursor += count
                
                # Sort
                ranked_results = sorted(zip(q_scores, doc_ids), key=lambda x: x[0], reverse=True)
                reranked_ids = [d_id for _, d_id in ranked_results]
                
                # Đánh giá Reranker
                if gt_ids:
                    for k in k_values:
                        h, r = calculate_metrics(reranked_ids, gt_ids, k)
                        metrics_reranker['hit'][k] += h
                        metrics_reranker['recall'][k] += r

    # Print final results
    print("\n" + "="*30)
    print("FINAL RESULTS")
    print("="*30)
    
    def print_pretty(name, metrics):
        print(f"\n>> {name}:")
        print(f"{'K':<5} | {'Hit Rate':<10} | {'Recall':<10}")
        print("-" * 30)
        for k in k_values:
            hit = metrics['hit'][k] / total_queries
            recall = metrics['recall'][k] / total_queries
            print(f"{k:<5} | {hit:.4f}     | {recall:.4f}")

    print_pretty("Stage 1: Bi-Encoder (Retrieval)", metrics_retriever)
    print_pretty("Stage 2: Cross-Encoder (Reranking)", metrics_reranker)

In [ ]:
if __name__ == "__main__":
    BASE_INPUT_DIR = '/kaggle/input'
    TEST_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/crawled_test_5k.jsonl')
    CORPUS_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/corpus.csv')
    FAISS_INDEX_PATH = os.path.join(BASE_INPUT_DIR, 'qa-retrieval-rag-chunking/index.faiss')
    evaluate_retrieval_rag_final_optimized(test_data_path=TEST_DATA_PATH, corpus_data_path=CORPUS_DATA_PATH, faiss_index_path=FAISS_INDEX_PATH)